# MoWM — Extract NV-Embed-v2 Embeddings for Zero-Shot VSA Projection

This notebook extracts embeddings from **nvidia/NV-Embed-v2** (4096D, #1 MTEB) and saves them as a domain codebook for MoWM's `SequenceEncoder.encode_embedded_tokens()`.

**Why NV-Embed-v2 over Qwen hidden states:**
- Purpose-built embedding model (not hidden-state extraction from a generative model)
- 4096D output — richer than Qwen2.5-7B's 3584D hidden states
- #1 on MTEB leaderboard — state-of-the-art retrieval, classification, clustering
- Instruction-tuned — supports task-prefixed queries for domain-specific encoding
- Bidirectional attention — captures full context (generative models are causal-only)

**Hardware:** Colab A100 (96GB VRAM) — runs NV-Embed-v2 at full precision

**Process:**
1. Load NV-Embed-v2 on Colab A100
2. Extract embeddings from training data with task instructions
3. Group by domain, compute centroids
4. Save as `.npz` codebook for MoWM

**Output:** `mowm_nvembed_codebook.npz` with:
- `embeddings`: (N, 4096) float32 array of all embeddings
- `texts`: list of source texts
- `domains`: list of domain labels per embedding
- `centroids`: (n_domains, 4096) domain centroid embeddings
- `domain_names`: list of domain names matching centroid rows

## 1. Setup & Install

In [ ]:
!pip install -q transformers torch accelerate sentencepiece
!pip install -q datasets einops  # einops needed for NV-Embed-v2

In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Load NV-Embed-v2

nvidia/NV-Embed-v2 outputs 4096D embeddings. On A100 96GB it runs at full bf16 precision.
The model uses a latent attention layer on top of a Mistral-7B backbone — bidirectional attention
means it sees the full context (unlike causal LLMs).

In [ ]:
MODEL_ID = "nvidia/NV-Embed-v2"
LATENT_DIM = 4096  # NV-Embed-v2 output dimension

print(f"Loading {MODEL_ID}...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

# NV-Embed-v2 uses the same tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

# Verify embedding dimension
print(f"Model loaded on {model.device}")
print(f"Expected embedding dim: {LATENT_DIM}")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 3. Embedding Extraction Function

NV-Embed-v2 supports **instruction-prefixed** encoding. Each query is prefixed with a task
instruction that tells the model what kind of embedding to produce:
- `"Instruct: Classify this text into a scientific domain\nQuery: ..."` for domain anchors
- `"Instruct: Represent this text for semantic retrieval\nQuery: ..."` for general text

This gives better domain separation than generic mean-pooling.

In [ ]:
# NV-Embed-v2 task instructions for different encoding modes
TASK_CLASSIFY = "Instruct: Classify this text by its knowledge domain\nQuery: "
TASK_RETRIEVE = "Instruct: Represent this text for semantic retrieval\nQuery: "
TASK_CLUSTER = "Instruct: Identify the core concept in this text\nQuery: "


@torch.no_grad()
def extract_embedding(
    text: str,
    task_instruction: str = TASK_RETRIEVE,
    max_length: int = 512,
) -> np.ndarray:
    """Extract embedding from NV-Embed-v2 with task instruction.

    Args:
        text: Input text string.
        task_instruction: Instruction prefix for the embedding task.
        max_length: Maximum token length (truncate longer).

    Returns:
        (LATENT_DIM,) float32 numpy array, L2-normalized.
    """
    # NV-Embed-v2 expects instruction + query format
    full_text = task_instruction + text

    inputs = tokenizer(
        full_text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
        padding=False,
    ).to(model.device)

    outputs = model(
        **inputs,
        output_hidden_states=True,
        return_dict=True,
    )

    # NV-Embed-v2: use last hidden state with mean pooling
    # (the model's latent attention layer produces the final embedding)
    last_hidden = outputs.hidden_states[-1]  # (1, seq_len, hidden_dim)

    # Mean pool over all tokens
    attention_mask = inputs["attention_mask"].unsqueeze(-1)
    masked = last_hidden * attention_mask
    pooled = masked.sum(dim=1) / attention_mask.sum(dim=1).clamp(min=1)

    # L2 normalize
    embedding = pooled[0].float().cpu().numpy()
    norm = np.linalg.norm(embedding)
    if norm > 0:
        embedding = embedding / norm

    return embedding.astype(np.float32)


@torch.no_grad()
def extract_embeddings_batch(
    texts: list[str],
    task_instruction: str = TASK_RETRIEVE,
    batch_size: int = 16,
    max_length: int = 512,
) -> np.ndarray:
    """Extract embeddings for a batch of texts.

    Args:
        texts: List of input strings.
        task_instruction: Instruction prefix for all texts.
        batch_size: Process this many at once (A100 can handle 16+).
        max_length: Maximum token length.

    Returns:
        (N, LATENT_DIM) float32 numpy array.
    """
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch_texts = [task_instruction + t for t in texts[i:i + batch_size]]
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            truncation=True,
            max_length=max_length,
            padding=True,
        ).to(model.device)

        outputs = model(
            **inputs,
            output_hidden_states=True,
            return_dict=True,
        )

        last_hidden = outputs.hidden_states[-1]
        attention_mask = inputs["attention_mask"].unsqueeze(-1)
        masked = last_hidden * attention_mask
        pooled = masked.sum(dim=1) / attention_mask.sum(dim=1).clamp(min=1)

        embeddings = pooled.float().cpu().numpy()
        norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
        norms = np.where(norms == 0, 1.0, norms)
        embeddings = (embeddings / norms).astype(np.float32)

        all_embeddings.append(embeddings)

        if (i // batch_size) % 25 == 0:
            print(f"  Processed {i + len(batch_texts)}/{len(texts)} texts")

    return np.concatenate(all_embeddings, axis=0)


# Quick test — verify dimension
test_emb = extract_embedding("Force equals mass times acceleration")
print(f"Test embedding shape: {test_emb.shape}")
print(f"Expected: ({LATENT_DIM},)")
print(f"Norm: {np.linalg.norm(test_emb):.4f}")
assert test_emb.shape == (LATENT_DIM,), f"Wrong dim: {test_emb.shape}"

# Quick semantic test
emb_physics = extract_embedding("Newton's law of motion", TASK_CLASSIFY)
emb_music = extract_embedding("Bach fugue in D minor", TASK_CLASSIFY)
emb_physics2 = extract_embedding("Gravitational acceleration", TASK_CLASSIFY)
sim_same = float(emb_physics @ emb_physics2)
sim_diff = float(emb_physics @ emb_music)
print(f"\nSemantic test:")
print(f"  physics-physics sim: {sim_same:.3f} (should be > 0.7)")
print(f"  physics-music sim:   {sim_diff:.3f} (should be < 0.5)")

## 4. Mount Google Drive (for data access)

Upload your training data to Google Drive, or use the cells below to load from URLs/local paths.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Set paths — update these to match your Drive structure
DRIVE_BASE = "/content/drive/MyDrive/mowm_data"

# Training data files (upload these to Drive first)
INSTRUCT_PATH = f"{DRIVE_BASE}/instruct_55k_clean.jsonl"
SVC_INSTRUCT_PATH = f"{DRIVE_BASE}/instruct_svc_semantic.jsonl"
SVC_CONVO_PATH = f"{DRIVE_BASE}/conversations_svc_semantic.jsonl"
EMOTIONS_PATH = f"{DRIVE_BASE}/emotions.jsonl"

# Output path — NV-Embed-v2 codebook
OUTPUT_PATH = f"{DRIVE_BASE}/mowm_nvembed_codebook.npz"

print(f"Drive mounted. Data base: {DRIVE_BASE}")
for p in [INSTRUCT_PATH, SVC_INSTRUCT_PATH, SVC_CONVO_PATH, EMOTIONS_PATH]:
    exists = os.path.exists(p)
    print(f"  {'OK' if exists else 'MISSING'}: {os.path.basename(p)}")

## 5. Load & Prepare Training Data by Domain

In [ ]:
def load_jsonl(path: str, max_records: int = 5000) -> list[dict]:
    """Load a JSONL file, return list of dicts."""
    records = []
    if not os.path.exists(path):
        print(f"  SKIP (not found): {path}")
        return records
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if len(records) >= max_records:
                break
            line = line.strip()
            if line:
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError:
                    continue
    print(f"  Loaded {len(records)} records from {os.path.basename(path)}")
    return records


def extract_text(record: dict) -> str:
    """Extract text from a training record (handles various formats)."""
    # Try common field names
    for key in ["text", "instruction", "input", "content", "question", "prompt"]:
        if key in record and record[key]:
            val = record[key]
            if isinstance(val, str) and len(val) > 10:
                return val[:1000]  # cap at 1000 chars
    # Fallback: concatenate all string values
    parts = [str(v) for v in record.values() if isinstance(v, str) and len(str(v)) > 5]
    return " ".join(parts)[:1000] if parts else ""

In [ ]:
# Domain-specific axiom sentences for anchor embeddings
# These are the "core concepts" that define each domain in embedding space
DOMAIN_ANCHORS: dict[str, list[str]] = {
    "physics": [
        "Force equals mass times acceleration, F = m * a",
        "Energy is conserved in isolated systems",
        "The velocity of an object changes when a net force acts on it",
        "Gravitational force between two masses is proportional to their product",
        "Kinetic energy equals one half mass times velocity squared",
        "Pressure equals force divided by area",
        "The wave equation relates frequency, wavelength, and velocity",
        "Ohm's law states voltage equals current times resistance",
    ],
    "logic": [
        "If A implies B and A is true, then B must be true (modus ponens)",
        "A proposition cannot be both true and false at the same time",
        "De Morgan's law: NOT(A AND B) equals NOT A OR NOT B",
        "If an obligation exists and no exception applies, the action is required",
        "A valid deductive argument preserves truth from premises to conclusion",
    ],
    "economics": [
        "Supply and demand determine market equilibrium price",
        "Nash equilibrium is where no player benefits from changing strategy alone",
        "GDP equals consumption plus investment plus government spending plus net exports",
        "The Fisher equation relates nominal interest rate to real rate and inflation",
        "Marginal utility decreases as consumption increases",
    ],
    "social": [
        "PageRank assigns importance based on incoming links from important pages",
        "Network diffusion models how information spreads through social connections",
        "Preferential attachment means popular nodes attract more connections",
        "The SIR model tracks susceptible, infected, and recovered populations",
    ],
    "causal": [
        "Bayes theorem computes posterior probability from prior and likelihood",
        "Causal intervention do(X) removes incoming edges to X in the causal graph",
        "Counterfactual reasoning asks what would have happened if X were different",
        "Granger causality tests whether past values of X improve prediction of Y",
    ],
    "atomic": [
        "The Schrodinger equation describes quantum state evolution over time",
        "Heisenberg uncertainty principle limits simultaneous measurement of position and momentum",
        "Electrons occupy discrete energy levels in atoms",
        "The Lennard-Jones potential models intermolecular attraction and repulsion",
    ],
    "bio": [
        "Hardy-Weinberg equilibrium describes allele frequencies in stable populations",
        "Hamilton's rule for altruism: relatedness times benefit exceeds cost",
        "DNA encodes genetic information through nucleotide base pairs A-T and G-C",
        "Natural selection favors traits that increase reproductive fitness",
        "Michaelis-Menten kinetics describe enzyme reaction rates",
    ],
    "music": [
        "Harmonic series frequencies are integer multiples of the fundamental",
        "Equal temperament divides the octave into twelve equal semitones",
        "Consonance correlates with simple frequency ratios between notes",
        "Fourier decomposition represents any signal as a sum of sinusoids",
    ],
    "identity": [
        "Personal identity is shaped by core values, beliefs, and experiences",
        "Emotional baseline varies across eight primary emotions in Plutchik's model",
        "Family bonds and cultural heritage form the foundation of personal identity",
        "Habits strengthen through repetition following an exponential learning curve",
    ],
    "colors": [
        "Visible light wavelength determines perceived color from violet to red",
        "RGB additive color mixing combines red green and blue light",
        "Complementary colors are opposite on the color wheel",
        "Color temperature in Kelvin describes warm to cool light appearance",
    ],
    "language": [
        "Sentence structure follows constituency grammar with noun and verb phrases",
        "Word meaning is determined by distributional context in large corpora",
        "Morphemes combine to form words through inflection and derivation",
        "Zipf's law relates word frequency to its rank in the distribution",
    ],
    "historical": [
        "Historical events form causal chains where earlier events enable later ones",
        "Civilizations follow cycles of rise, peak, decline, and renewal",
        "Technology adoption follows an S-curve from early adopters to mainstream",
        "Counterfactual history explores what might have happened differently",
    ],
    "ensemble": [
        "Population behavior emerges from aggregating many individual decisions",
        "Mean-field approximation replaces individual interactions with average effect",
        "Phase transitions occur when system parameters cross critical thresholds",
        "Wisdom of crowds: aggregate estimates converge to true values",
    ],
    "linguistics": [
        "Context-free grammars generate sentences through production rules",
        "Dependency grammar represents syntactic structure through head-dependent relations",
        "Binding theory constrains the interpretation of pronouns and anaphors",
    ],
}

print(f"Domain anchors: {len(DOMAIN_ANCHORS)} domains")
total_anchors = sum(len(v) for v in DOMAIN_ANCHORS.values())
print(f"Total anchor sentences: {total_anchors}")

In [ ]:
# Load training data from JSONL files
print("Loading training data...")

all_texts: list[str] = []
all_domains: list[str] = []

# 1. Domain anchors (highest priority — these define the domain centers)
for domain, sentences in DOMAIN_ANCHORS.items():
    for sent in sentences:
        all_texts.append(sent)
        all_domains.append(domain)
print(f"Anchors: {len(all_texts)} texts")

# 2. Instruction data (general coverage)
instruct_records = load_jsonl(INSTRUCT_PATH, max_records=3000)
for rec in instruct_records:
    text = extract_text(rec)
    if text:
        all_texts.append(text)
        all_domains.append("general")  # will be classified later

# 3. SVC instruction data (has domain labels)
svc_instruct = load_jsonl(SVC_INSTRUCT_PATH, max_records=2000)
for rec in svc_instruct:
    text = extract_text(rec)
    domain = rec.get("domain", rec.get("category", "general"))
    if isinstance(domain, str):
        domain = domain.lower().split(".")[0]
    if text:
        all_texts.append(text)
        all_domains.append(str(domain))

# 4. SVC conversation data
svc_convo = load_jsonl(SVC_CONVO_PATH, max_records=1000)
for rec in svc_convo:
    text = extract_text(rec)
    domain = rec.get("domain", rec.get("category", "general"))
    if isinstance(domain, str):
        domain = domain.lower().split(".")[0]
    if text:
        all_texts.append(text)
        all_domains.append(str(domain))

# 5. Emotion data (identity domain)
emotion_records = load_jsonl(EMOTIONS_PATH, max_records=1000)
for rec in emotion_records:
    text = rec.get("text", "")
    if text:
        all_texts.append(text)
        all_domains.append("identity")

print(f"\nTotal texts to embed: {len(all_texts)}")
print(f"Domain distribution:")
from collections import Counter
domain_counts = Counter(all_domains)
for domain, count in domain_counts.most_common(20):
    print(f"  {domain}: {count}")

## 6. Extract Embeddings

In [ ]:
print(f"Extracting {len(all_texts)} embeddings with NV-Embed-v2...")
print(f"Batch size: 16 (A100 96GB), max_length: 512")
print(f"Estimated time: ~{len(all_texts) / 16 * 0.3 / 60:.0f} minutes on A100\n")

# Separate anchors (use TASK_CLASSIFY) from general text (use TASK_RETRIEVE)
n_anchors = total_anchors
anchor_texts = all_texts[:n_anchors]
general_texts = all_texts[n_anchors:]

print(f"Phase 1: {len(anchor_texts)} domain anchors with TASK_CLASSIFY...")
anchor_embeddings = extract_embeddings_batch(
    anchor_texts,
    task_instruction=TASK_CLASSIFY,
    batch_size=16,
    max_length=512,
)

print(f"\nPhase 2: {len(general_texts)} general texts with TASK_RETRIEVE...")
general_embeddings = extract_embeddings_batch(
    general_texts,
    task_instruction=TASK_RETRIEVE,
    batch_size=16,
    max_length=512,
)

embeddings = np.concatenate([anchor_embeddings, general_embeddings], axis=0)

print(f"\nDone! Embeddings shape: {embeddings.shape}")
print(f"Dtype: {embeddings.dtype}")
print(f"Mean norm: {np.linalg.norm(embeddings, axis=1).mean():.4f}")
print(f"VRAM peak: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

## 7. Compute Domain Centroids

In [ ]:
# Group embeddings by domain and compute centroids
domain_names_unique = sorted(set(all_domains))
centroids: dict[str, np.ndarray] = {}

print("Computing domain centroids...\n")
for domain in domain_names_unique:
    indices = [i for i, d in enumerate(all_domains) if d == domain]
    if not indices:
        continue
    domain_embs = embeddings[indices]
    centroid = domain_embs.mean(axis=0)
    # L2 normalize
    norm = np.linalg.norm(centroid)
    if norm > 0:
        centroid = centroid / norm
    centroids[domain] = centroid.astype(np.float32)
    print(f"  {domain}: {len(indices)} embeddings -> centroid norm {np.linalg.norm(centroid):.4f}")

print(f"\nTotal centroids: {len(centroids)}")

In [ ]:
# Verify: domain centroids should be distinguishable
print("Cross-domain cosine similarity (should be < 0.9 for different domains):\n")
mowm_domains = [
    "physics", "logic", "economics", "social", "causal",
    "atomic", "bio", "music", "identity", "colors",
    "language", "historical", "ensemble",
]
available_domains = [d for d in mowm_domains if d in centroids]

# Print similarity matrix header
header = "         " + " ".join(f"{d[:6]:>6}" for d in available_domains)
print(header)

for d1 in available_domains:
    row = f"{d1[:8]:>8} "
    for d2 in available_domains:
        sim = float(centroids[d1] @ centroids[d2])
        row += f"{sim:6.3f} "
    print(row)

## 8. Multilingual Verification

Verify that Qwen2.5's multilingual embeddings preserve cross-language similarity.

In [ ]:
# Test multilingual alignment
multilingual_pairs = [
    ("Force equals mass times acceleration", "La force est egale a la masse fois l'acceleration"),
    ("Force equals mass times acceleration", "力等于质量乘以加速度"),
    ("Energy is conserved", "L'energie est conservee"),
    ("Energy is conserved", "エネルギーは保存される"),
    ("Supply and demand", "Offre et demande"),
    ("Family is important", "La famille est importante"),
    ("Family is important", "家族は大切です"),
]

print("Multilingual cosine similarities (should be > 0.7):\n")
for en, other in multilingual_pairs:
    emb_en = extract_embedding(en)
    emb_other = extract_embedding(other)
    sim = float(emb_en @ emb_other)
    lang = "FR" if any(c in other for c in "éèêë") else (
        "ZH" if any('\u4e00' <= c <= '\u9fff' for c in other) else (
            "JA" if any('\u3040' <= c <= '\u30ff' for c in other) else "??"
        )
    )
    print(f"  EN-{lang} sim={sim:.3f}  |  '{en[:40]}' vs '{other[:40]}'")

## 9. Save Codebook

In [ ]:
# Prepare centroid matrix
centroid_names = sorted(centroids.keys())
centroid_matrix = np.stack([centroids[d] for d in centroid_names], axis=0)

print(f"Saving codebook to {OUTPUT_PATH}...")
print(f"  embeddings: {embeddings.shape}")
print(f"  centroids:  {centroid_matrix.shape}")
print(f"  domains:    {len(centroid_names)} unique")

np.savez_compressed(
    OUTPUT_PATH,
    embeddings=embeddings,
    texts=np.array(all_texts, dtype=object),
    domains=np.array(all_domains, dtype=object),
    centroids=centroid_matrix,
    domain_names=np.array(centroid_names, dtype=object),
    model_id=MODEL_ID,
    latent_dim=LATENT_DIM,
)

file_size_mb = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)
print(f"\nSaved! File size: {file_size_mb:.1f} MB")

## 10. Verify: Load and Use with MoWM SequenceEncoder

This cell shows how to use the codebook with MoWM (run locally, not on Colab).

In [ ]:
# Example: how to use the codebook with MoWM (run this locally)
print("""
# ── Usage with MoWM (run locally) ──────────────────────────

import numpy as np
from mowm.sequence_encoder import SequenceEncoder

# Load the NV-Embed-v2 codebook
codebook = np.load("mowm_nvembed_codebook.npz", allow_pickle=True)
latent_dim = int(codebook["latent_dim"])  # 4096
centroids = codebook["centroids"]         # (n_domains, 4096)
domain_names = codebook["domain_names"]   # ['atomic', 'bio', 'causal', ...]
embeddings = codebook["embeddings"]       # (N, 4096)

# Create encoder with orthogonal projection matrix for 4096D
enc = SequenceEncoder(k=80, l=128, latent_dim=latent_dim, seed=42)

# Project a domain centroid into VSA space
physics_idx = list(domain_names).index("physics")
physics_vsa = enc._project_embedding(centroids[physics_idx])
print(f"Physics VSA vector: {physics_vsa.shape}")  # (80, 128)

# Encode a sequence of NV-Embed-v2 embeddings
new_embeddings = [embeddings[0], embeddings[1], embeddings[2]]
sequence_vec = enc.encode_embedded_tokens(new_embeddings)
print(f"Sequence VSA vector: {sequence_vec.shape}")  # (80, 128)

# Similarity between projected embeddings preserves NV-Embed-v2 semantics!
from cubemind.ops.block_codes import BlockCodes
bc = BlockCodes(k=80, l=128)
v_physics = enc._project_embedding(centroids[physics_idx])
v_atomic = enc._project_embedding(centroids[list(domain_names).index("atomic")])
v_music = enc._project_embedding(centroids[list(domain_names).index("music")])
print(f"Physics-Atomic similarity: {bc.similarity(v_physics, v_atomic):.3f}")  # high
print(f"Physics-Music similarity:  {bc.similarity(v_physics, v_music):.3f}")   # low
""")

In [ ]:
print("Done! Download mowm_nvembed_codebook.npz from Google Drive.")
print(f"File: {OUTPUT_PATH}")
print(f"Size: {file_size_mb:.1f} MB")
print(f"Contains {embeddings.shape[0]} embeddings x {LATENT_DIM}D from {MODEL_ID}")
print(f"\nTo use locally:")
print(f"  SequenceEncoder(k=80, l=128, latent_dim={LATENT_DIM})")
print(f"  enc.encode_embedded_tokens([emb1, emb2, ...])  # zero-shot!")